<a href="https://colab.research.google.com/github/LionardoGomiz/Apache-Spark/blob/L/Funciones_En_Spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!apt-get -y install openjdk-17-jdk-headless

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] += ":/usr/lib/jvm/java-17-openjdk-amd64/bin"

import subprocess
print(subprocess.check_output(["java","-version"], stderr=subprocess.STDOUT).decode())
print("JAVA_HOME =", os.environ["JAVA_HOME"])


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-17-jdk-headless is already the newest version (17.0.16+8~us1-0ubuntu1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.
openjdk version "17.0.16" 2025-07-15
OpenJDK Runtime Environment (build 17.0.16+8-Ubuntu-0ubuntu122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.16+8-Ubuntu-0ubuntu122.04.1, mixed mode, sharing)

JAVA_HOME = /usr/lib/jvm/java-17-openjdk-amd64


In [12]:
!pip install -q findspark

In [13]:
!pip install -q pyspark

# Fechas y horas en PySpark (DataFrames)

### ¿Por qué tratar fechas/horas explícitamente?

En muchos datasets, las fechas u horas llegan como **textos** (string). Para poder **calcular, comparar, agrupar o formatear** correctamente necesitamos convertir esos textos en tipos nativos: **DateType (fecha)** y **TimestampType (fecha-hora)**. En PySpark esto se hace con funciones de la API *pyspark.sql.functions*.

### Funciones clave que usaremos

- **to_date(col, format=None):** Convierte una columna a tipo fecha. Si indicamos **format**, usamos un patrón como **dd-MM-yyyy**; si no, aplica reglas de conversión por defecto.

- **to_timestamp(col, format):** Convierte a marca de tiempo **(timestamp)**, aceptando también un patrón.

- **date_format():** Devuelve un **string formateado** a partir de una fecha/timestamp. Esto produce un texto.


In [16]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [17]:
data = spark.read.parquet('./data/convertir')

In [18]:
data.printSchema()

root
 |-- date: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- date_str: string (nullable = true)
 |-- ts_str: string (nullable = true)



In [19]:
data.show(truncate=False)

+----------+-----------------------+----------+----------------+
|date      |timestamp              |date_str  |ts_str          |
+----------+-----------------------+----------+----------------+
|2021-01-01|2021-01-01 20:10:50.723|01-01-2021|18-08-2021 46:58|
+----------+-----------------------+----------+----------------+



In [20]:
from pyspark.sql.functions import col, to_date, to_timestamp

In [21]:
data1 = data.select(
    to_date(col('date')).alias('date1'),
    to_timestamp(col('timestamp')).alias('ts1'),
    to_date(col('date_str'), 'dd-MM-yyyy').alias('date2'),
    to_timestamp(col('ts_str'), 'dd-MM-yyyy mm:ss').alias('ts2')
)


In [ ]:
data1.show(truncate=False)
data1.printSchema()

In [24]:
from pyspark.sql.functions import date_format

In [27]:
data1.select(
    date_format(col('date1'), 'dd--MM-yyyy')
).show()

+-------------------------------+
|date_format(date1, dd--MM-yyyy)|
+-------------------------------+
|                    01--01-2021|
+-------------------------------+



In [28]:
df = spark.read.parquet('./data/calculo.parquet')

In [29]:
df.show()

+------+-------------+------------+-------------------+
|nombre|fecha_ingreso|fecha_salida|       baja_sistema|
+------+-------------+------------+-------------------+
|  Jose|   2021-01-01|  2021-11-14|2021-10-14 15:35:59|
|Mayara|   2021-02-06|  2021-11-25|2021-11-25 10:35:55|
+------+-------------+------------+-------------------+



In [32]:
from pyspark.sql.functions import datediff, months_between, last_day

In [37]:
df.select(
    col('nombre'),
    datediff(col('fecha_salida'), col('fecha_ingreso')).alias('Dias'),
    months_between(col('fecha_salida'), col('fecha_ingreso')).alias('meses'),
    last_day(col('fecha_salida')).alias('ultimo_dia_mes')
).show()

+------+----+-----------+--------------+
|nombre|Dias|      meses|ultimo_dia_mes|
+------+----+-----------+--------------+
|  Jose| 317|10.41935484|    2021-11-30|
|Mayara| 292| 9.61290323|    2021-11-30|
+------+----+-----------+--------------+



In [38]:
from pyspark.sql.functions import date_add, date_sub

In [40]:
df.select(
    col('nombre'),
    col('fecha_ingreso'),
    date_add(col('fecha_ingreso'), 15).alias('mas_14_dias'),
    date_sub(col('fecha_ingreso'), 1).alias('menos_1_dia')
).show()


+------+-------------+-----------+-----------+
|nombre|fecha_ingreso|mas_14_dias|menos_1_dia|
+------+-------------+-----------+-----------+
|  Jose|   2021-01-01| 2021-01-16| 2020-12-31|
|Mayara|   2021-02-06| 2021-02-21| 2021-02-05|
+------+-------------+-----------+-----------+



In [41]:
from pyspark.sql.functions import year, month, dayofmonth, dayofyear, hour, minute, second


In [42]:
df.select(
    col('baja_sistema'),
    year(col('baja_sistema')),
    month(col('baja_sistema')),
    dayofmonth(col('baja_sistema')),
    dayofyear(col('baja_sistema')),
    hour(col('baja_sistema')),
    minute(col('baja_sistema')),
    second(col('baja_sistema'))
).show()


+-------------------+------------------+-------------------+------------------------+-----------------------+------------------+--------------------+--------------------+
|       baja_sistema|year(baja_sistema)|month(baja_sistema)|dayofmonth(baja_sistema)|dayofyear(baja_sistema)|hour(baja_sistema)|minute(baja_sistema)|second(baja_sistema)|
+-------------------+------------------+-------------------+------------------------+-----------------------+------------------+--------------------+--------------------+
|2021-10-14 15:35:59|              2021|                 10|                      14|                    287|                15|                  35|                  59|
|2021-11-25 10:35:55|              2021|                 11|                      25|                    329|                10|                  35|                  55|
+-------------------+------------------+-------------------+------------------------+-----------------------+------------------+-----------------

# Limpieza y manipulación de textos
(string) con PySpark

### ¿Por qué manipular strings en big data?

En datos **reales**, los campos de texto traen **espacios en blanco,
capitalización inconsistente, longitudes variables y patrones** que
queremos normalizar o sustituir.
PySpark ofrece funciones vectorizadas para resolver esto con eficiencia.

### Funciones clave usadas:

- **trim/ltrim/rtrim:** Eliminan
espacios en **ambos** extremos
(*trim*), solo **izquierdo** (ltrim)
o solo **derecho** (rtrim) de un string.

Desde Spark 4, opcionalmente aceptan el conjunto de caracteres a recortar.

lpad / rpad: Rellena una cadena a una longitud fija agregando caracteres a la izquierda o derecha. Útil para código con ceros a la izquierda, formatos uniformes, etc